**Numeric constraints**

In [ ]:
from pydantic import BaseModel, Field
from decimal import Decimal

class TrainingPydantic(BaseModel):
    id: int = Field(..., gt=0, le=1000, description="The ID must be greater than 0")
    batch_size: int = Field(gt=0, multiple_of=8)
    learning_rate: float = Field(gt=0.0, lt=1.0, allow_inf_nan=False)
    budget_used: Decimal = Field(gt=0, max_digits=8, decimal_places=2)

In [7]:
#valid
t = TrainingPydantic(id=10,batch_size=32, learning_rate=0.001, budget_used=150.50)    
print(t)

id=10 batch_size=32 learning_rate=0.001 budget_used=Decimal('150.5')


In [9]:
# Invalid
try:
    t = TrainingPydantic(id=10,batch_size=15, learning_rate=2.001, budget_used=150.50)
except Exception as e:
    print(e.errors())
    for error in e.errors():
        print(f"Error in field '{error['loc'][0]}' --> {error['msg']}")

[{'type': 'multiple_of', 'loc': ('batch_size',), 'msg': 'Input should be a multiple of 8', 'input': 15, 'ctx': {'multiple_of': 8}, 'url': 'https://errors.pydantic.dev/2.14/v/multiple_of'}, {'type': 'less_than', 'loc': ('learning_rate',), 'msg': 'Input should be less than 1', 'input': 2.001, 'ctx': {'lt': 1.0}, 'url': 'https://errors.pydantic.dev/2.14/v/less_than'}]
Error in field 'batch_size' --> Input should be a multiple of 8
Error in field 'learning_rate' --> Input should be less than 1


**string constraints**

In [20]:
from typing import Annotated
from pydantic import BaseModel, Field, ValidationError, StringConstraints

CleanStringConstraint = Annotated[str, StringConstraints(strip_whitespace=True, min_length=1)]
RollNumberVal = Annotated[str, StringConstraints(strip_whitespace=True, to_upper=True, pattern=r'^[A-Za-z]{2,5}-\d{3}$')]

class BWTClass(BaseModel):
    name: CleanStringConstraint
    roll_no: RollNumberVal
    
# valid scenario
b = BWTClass(name="  John Doe  ", roll_no="ABC-123")
print(b)  

name='John Doe' roll_no='ABC-123'


In [21]:
# invalid scenario
try:
    b = BWTClass(name="   ", roll_no="cjdcuwihd")
except ValidationError as e:
    print(e.errors())
    for error in e.errors():
        print(f"Error in field '{error['loc'][0]}' --> {error['msg']}")

[{'type': 'string_too_short', 'loc': ('name',), 'msg': 'String should have at least 1 character', 'input': '   ', 'ctx': {'min_length': 1}, 'url': 'https://errors.pydantic.dev/2.14/v/string_too_short'}, {'type': 'string_pattern_mismatch', 'loc': ('roll_no',), 'msg': "String should match pattern '^[A-Za-z]{2,5}-\\d{3}$'", 'input': 'cjdcuwihd', 'ctx': {'pattern': '^[A-Za-z]{2,5}-\\d{3}$'}, 'url': 'https://errors.pydantic.dev/2.14/v/string_pattern_mismatch'}]
Error in field 'name' --> String should have at least 1 character
Error in field 'roll_no' --> String should match pattern '^[A-Za-z]{2,5}-\d{3}$'


**list constraints**

In [25]:
from pydantic import BaseModel, PositiveInt

ProbVal = Annotated[float, Field(gt=0.0, lt=1.0, allow_inf_nan=False)]
class NewClass(BaseModel):
    request_id: PositiveInt
    class_prob: list[ProbVal] = Field(min_length=2, max_length=10)
    embedding: list[ProbVal] = Field(min_length=3, max_length=6, repr=False)
    
out = NewClass(request_id=101, class_prob=[0.8, 0.5], embedding=[0.12, 0.44, 0.44])
print(out)
print(out.embedding)

inn = out
print(inn)
print(inn.embedding)

request_id=101 class_prob=[0.8, 0.5]
[0.12, 0.44, 0.44]
request_id=101 class_prob=[0.8, 0.5]
[0.12, 0.44, 0.44]


In [29]:
# Invalid
try:
    NewClass(request_id=-1101, class_prob=[1.5, 0.8], embedding=[0.12, 0.44])
except ValidationError as e:
    print(e.error_count())
    for err in e.errors():
        print(f"Error in field '{err['loc'][0]}' --> {err['msg']}")

3
Error in field 'request_id' --> Input should be greater than 0
Error in field 'class_prob' --> Input should be less than 1
Error in field 'embedding' --> List should have at least 3 items after validation, not 2


**Strict and Frozen:**

In [ ]:
from pydantic import BaseModel, Field, ValidationError
from typing import Annotated

class StrictUser(BaseModel): # frozen=True makes the model immutable, meaning that once an instance is created, its attributes cannot be modified. This is useful for creating read-only data structures.
    # strict forces to type check the user_id field and make it immutable (frozen).
    # do not change also
    user_id: int = Field(strict=True, frozen=True)
    val: int
    
u = StrictUser(user_id=10, val="10")
print(u)

user_id=10 val=10


In [ ]:
from pydantic import BaseModel, Field

class Session(BaseModel):
    username: str
    api_key: str = Field(exclude=True) # the api_key field should be excluded from certain operations, such as serialization or model dumping. This is useful for sensitive information that you don't want to expose when converting the model to a dictionary or JSON format.

s = Session(username="user1", api_key="secret_key")

print(s)
print(s.api_key)
print(s.model_dump()) # to convert the Pydantic model instance into a dictionary representation. It returns a dictionary containing the field names and their corresponding values, excluding any fields marked with exclude=True.

username='user1' api_key='secret_key'
secret_key
{'username': 'user1'}
